## Data Curation — Refactored Pipeline

Uses `ESCIProcessor`, `WANDSProcessor`, and `DatasetMerger` from  
`src/product_search/data_curation.py` to build a unified product dataset.

**Unified `ProductDocument` schema:**

| Field | Type | Notes |
|---|---|---|
| `product_id` | keyword | prefixed: `amz_<id>` / `wands_<id>` |
| `source` | keyword | `"ESCI"` \| `"WANDS"` |
| `full_text` | text | enriched for BM25: title > brand > bullets > desc |
| `encode_text` | text | short text for dense embedding: title + brand only |
| `brand` | keyword | normalised, lowercased (ESCI only) |
| `color` | keyword[] | normalised list, compound colours split |
| `product_class` | keyword | high-level category (WANDS only) |
| `category_path` | keyword | WANDS hierarchy; null for ESCI |
| `average_rating` | float | null for ESCI |
| `review_count` | integer | null for ESCI |
| `metadata` | object | full raw fields preserved for debugging |

In [8]:
import os
import sys
import pandas as pd
from pathlib import Path
from typing import Tuple

project_dir = Path(os.getcwd()).parent
data_dir    = project_dir / "Data" / "RAW"

# Make the src package importable
sys.path.insert(0, str(project_dir / "src"))

from product_search.data_curation import ESCIProcessor, WANDSProcessor, DatasetMerger

### 1. Load Amazon ESCI

In [9]:
examples_df = pd.read_parquet(data_dir / "shopping_queries_dataset_examples.parquet")
products_df = pd.read_parquet(data_dir / "shopping_queries_dataset_products.parquet")

amz_esci_df = pd.merge(
    examples_df, products_df,
    how="left", on="product_id",
    suffixes=("_example", "_product"),
)

# Keep only the small-version US locale rows (per dataset paper)
amz_esci_df = (
    amz_esci_df[amz_esci_df["small_version"] == 1]
    .drop(columns=["small_version", "large_version"])
)
amz_esci_df = (
    amz_esci_df[amz_esci_df["product_locale_example"] == "us"]
    .drop(columns=["product_locale_example", "product_locale_product"])
)

amz_train_df = amz_esci_df[amz_esci_df["split"] == "train"].drop(columns=["split"])
amz_test_df  = amz_esci_df[amz_esci_df["split"] == "test"].drop(columns=["split"])

print(f"ESCI train : {amz_train_df.shape}  |  unique queries: {amz_train_df['query_id'].nunique():,}")
print(f"ESCI test  : {amz_test_df.shape}  |  unique queries: {amz_test_df['query_id'].nunique():,}")

ESCI train : (427655, 10)  |  unique queries: 20,888
ESCI test  : (185361, 10)  |  unique queries: 8,956


### 2. Load Wayfair WANDS

In [10]:
def query_level_split(
    df: pd.DataFrame,
    test_size: float = 0.3,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Split by unique query_id to avoid train/test leakage."""
    qids = pd.Series(df["query_id"].astype(str).unique()).sample(frac=1.0, random_state=seed)
    cut  = int(len(qids) * (1 - test_size))
    train_qids = set(qids.iloc[:cut])
    test_qids  = set(qids.iloc[cut:])
    assert not train_qids & test_qids, "Overlapping query_ids in train/test!"
    return (
        df[df["query_id"].astype(str).isin(train_qids)].reset_index(drop=True),
        df[df["query_id"].astype(str).isin(test_qids)].reset_index(drop=True),
    )


product_df = pd.read_csv(data_dir / "product.csv", sep="\t")
query_df   = pd.read_csv(data_dir / "query.csv",   sep="\t")
label_df   = pd.read_csv(data_dir / "label.csv",   sep="\t")

wands_df = (
    pd.merge(query_df, label_df, on="query_id", how="inner")
      .merge(product_df, on="product_id", how="inner")
      .drop(columns=["id"])
)

wands_train_df, wands_test_df = query_level_split(wands_df, test_size=0.3, seed=42)

print(f"WANDS train: {wands_train_df.shape}  |  unique queries: {wands_train_df['query_id'].nunique():,}")
print(f"WANDS test : {wands_test_df.shape}  |  unique queries: {wands_test_df['query_id'].nunique():,}")

WANDS train: (162789, 13)  |  unique queries: 336
WANDS test : (70659, 13)  |  unique queries: 144


### 3. Build unified `MergedArtifacts` via `DatasetMerger`

In [11]:
esci  = ESCIProcessor(amz_train_df, amz_test_df)
wands = WANDSProcessor(wands_train_df, wands_test_df)
merger    = DatasetMerger([esci, wands])

# For Production
# artifacts = merger.build()

# Dev sample (2k train / 500 test / ~small product store)
artifacts = merger.build_dev_sample(
    n_train_per_source=500,      # 1000 ESCI + 1000 WANDS = 2000 train
    n_test_per_source=100,        # 250 ESCI + 250 WANDS = 500 test
    n_distractor_per_source=100,  # 250 ESCI + 250 WANDS = 500 extra noise products
    random_state=42,              # reproducible
)


[DatasetMerger] ── Merge summary ──────────────────────────
  Total products  :   23,052
    ESCI        :   10,390
    WANDS       :   12,662
  Train queries   :      835
  Test  queries   :      200
  Train qrel docs :   21,791
  Test  qrel docs :    5,443
─────────────────────────────────────────────────────────────



### 4. Final unified product DataFrame

In [12]:
df = pd.DataFrame.from_dict(artifacts.product_store, orient="index").reset_index(drop=True)

print(f"Total products              : {len(df):,}")
print(f"Source breakdown            : {df['source'].value_counts().to_dict()}")
print(f"Products with brand         : {df['brand'].notna().sum():,}")
print(f"Products with color info    : {df['color'].apply(bool).sum():,}")
print(f"Products with product_class : {df['product_class'].notna().sum():,}")
print(f"Products with rating        : {df['average_rating'].notna().sum():,}")
print()

display_cols = [
    "product_id", "source", "brand", "color",
    "product_class", "category_path",
    "average_rating", "review_count", "encode_text", "full_text",
]

Total products              : 23,052
Source breakdown            : {'WANDS': 12662, 'ESCI': 10390}
Products with brand         : 9,950
Products with color info    : 9,395
Products with product_class : 11,935
Products with rating        : 9,841



In [13]:
df[display_cols].head(n=3)

,product_id,source,brand,color,product_class,category_path,average_rating,review_count,encode_text,full_text
0,amz_B00Y1XEYFY,ESCI,frigidaire,[],NaN,None,NaN,NaN,Frigidaire 807117001 Drain Hose Frigidaire,Frigidaire 807117001 Drain Hose Frigidaire REC...
1,amz_B07FCTCSYW,ESCI,novogratz,[white],NaN,None,NaN,NaN,"Novogratz Computer Desk with Storage, White Ma...","Novogratz Computer Desk with Storage, White Ma..."
2,amz_B07GJWZYVQ,ESCI,cosmetasa,[],NaN,None,NaN,NaN,Anti-Cellulite Massage Oil & Hot Cream - 100% ...,Anti-Cellulite Massage Oil & Hot Cream - 100% ...


### 5. Save processed artifacts to `Data/PROCESSED/`

Writes the three files expected by `database_ingestion.ipynb`:

| File | Contents |
|---|---|
| `product_store.json` | `{product_id: ProductDocument-as-dict}` |
| `train_qrels.json` | `{query_id: {product_id: gain}}` |
| `test_qrels.json` | `{query_id: {product_id: gain}}` |

Also saves `train_queries.csv` / `test_queries.csv` for evaluation pipelines.

In [14]:
import json

processed_dir = project_dir / "Data" / "PROCESSED"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save product store (dict of serialised ProductDocuments)
with open(processed_dir / "product_store.json", "w", encoding="utf-8") as f:
    json.dump(artifacts.product_store, f, ensure_ascii=False)
print(f"Saved product_store.json  ({len(artifacts.product_store):,} products)")

# Save qrels
with open(processed_dir / "train_qrels.json", "w", encoding="utf-8") as f:
    json.dump(artifacts.train_qrels_dict, f, ensure_ascii=False)
print(f"Saved train_qrels.json    ({len(artifacts.train_qrels_dict):,} queries)")

with open(processed_dir / "test_qrels.json", "w", encoding="utf-8") as f:
    json.dump(artifacts.test_qrels_dict, f, ensure_ascii=False)
print(f"Saved test_qrels.json     ({len(artifacts.test_qrels_dict):,} queries)")

# Save query tables as CSV (useful for evaluation pipelines)
artifacts.train_query_table.to_csv(processed_dir / "train_queries.csv", index=False)
artifacts.test_query_table.to_csv(processed_dir / "test_queries.csv", index=False)
print(f"Saved train_queries.csv   ({len(artifacts.train_query_table):,} rows)")
print(f"Saved test_queries.csv    ({len(artifacts.test_query_table):,} rows)")

Saved product_store.json  (23,052 products)
Saved train_qrels.json    (835 queries)
Saved test_qrels.json     (200 queries)
Saved train_queries.csv   (835 rows)
Saved test_queries.csv    (200 rows)
